In [218]:
import pandas


In [219]:
df = pandas.read_parquet("/content/parquets")
label_df = pandas.read_parquet("/content/parquets")

In [220]:
label_df["DepDel15"]

,DepDel15
0,0.0
1,0.0
2,0.0
3,0.0
4,1.0
...,...
35189,1.0
35190,0.0
35191,1.0
35192,0.0


In [221]:
df.shape

(35194, 68)

In [222]:
df.columns

Index(['DestAirportID', 'OriginAirportID', 'Year', 'Month', 'DayofMonth',
       'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network',
       'DOT_ID_Marketing_Airline', 'Operating_Airline ',
       'DOT_ID_Operating_Airline', 'Flight_Number_Operating_Airline',
       'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName',
       'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName',
       'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
       'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff',
       'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay',
       'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk',
       'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay',
       'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
       'OriginICAO', 'OriginTimezone', 'DestICAO', 'DestTimezone',
       'CRSDepTimestamp', 'CRSArrTimestamp', 'OriginWindDirection',
       'OriginWindSpeed', 

In [223]:
missing_values = df.isnull().sum()
columns_with_nulls = missing_values[missing_values > 0]

if not columns_with_nulls.empty:
    print("Columns with null values and their counts:")
    display(columns_with_nulls)
else:
    print("No null values found in any column.")

Columns with null values and their counts:


,0
DepTime,478
DepDelay,478
DepDelayMinutes,478
DepDel15,478
DepartureDelayGroups,478
TaxiOut,496
WheelsOff,496
WheelsOn,499
TaxiIn,499
ArrTime,499


In [224]:
cols_to_drop = [
    # Actual departure/arrival values → leakage
    "DepTime", "DepDelay", "DepDelayMinutes",
    "DepartureDelayGroups", "TaxiOut", "WheelsOff", "WheelsOn",
    "TaxiIn", "ArrTime", "ArrDelay", "ArrDelayMinutes",
    "ArrDel15", "ArrivalDelayGroups",

    # After-fact outcomes
    "Cancelled", "Diverted",

    # Redundant ID fields
    "DestAirportID", "OriginAirportID", "DOT_ID_Marketing_Airline",
    "DOT_ID_Operating_Airline", "OriginAirportSeqID",
    "DestAirportSeqID", "OriginCityMarketID", "DestCityMarketID",
    "Flight_Number_Operating_Airline", "OriginICAO", "DestICAO",

    # Duplicated / unnecessary text fields
    "OriginCityName", "DestCityName",
    "DepTimeBlk", "ArrTimeBlk",
    "OriginTimezone", "DestTimezone",

    # Redundant timestamps
    "CRSDepTimestamp", "CRSArrTimestamp",

    # Columsn that have a lot of nan that seem useful
    'CancellationCode',
    'CarrierDelay','WeatherDelay','NASDelay',"SecurityDelay","LateAircraftDelay",
    "OriginWindGusts", "OriginTemperature", "OriginDewPoint",
    "DestWindGusts",'DestTemperature','DestDewPoint'
]

label_cols_to_drop = [
    # Actual departure/arrival values → leakage
    "DepTime",
    "DepartureDelayGroups", "TaxiOut", "WheelsOff", "WheelsOn",
    "TaxiIn", "ArrTime", "ArrivalDelayGroups",

    # After-fact outcomes
    "Cancelled", "Diverted",

    # Redundant ID fields
    "DestAirportID", "OriginAirportID", "DOT_ID_Marketing_Airline",
    "DOT_ID_Operating_Airline", "OriginAirportSeqID",
    "DestAirportSeqID", "OriginCityMarketID", "DestCityMarketID",
    "Flight_Number_Operating_Airline", "OriginICAO", "DestICAO",

    # Duplicated / unnecessary text fields
    "OriginCityName", "DestCityName",
    "DepTimeBlk", "ArrTimeBlk",
    "OriginTimezone", "DestTimezone",

    # Redundant timestamps
    "CRSDepTimestamp", "CRSArrTimestamp",

    # Columsn that have a lot of nan that seem useful
    'CancellationCode',
    'CarrierDelay','WeatherDelay','NASDelay',"SecurityDelay","LateAircraftDelay",
    "OriginWindGusts", "OriginTemperature", "OriginDewPoint",
    "DestWindGusts",'DestTemperature','DestDewPoint'
]

In [225]:
from numpy import column_stack
df.drop(columns=cols_to_drop, inplace=True)

In [226]:
from numpy import column_stack
label_df.drop(columns=label_cols_to_drop, inplace=True)

In [227]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35194 entries, 0 to 35193
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       35194 non-null  int32  
 1   Month                      35194 non-null  int32  
 2   DayofMonth                 35194 non-null  int32  
 3   DayOfWeek                  35194 non-null  int32  
 4   FlightDate                 35194 non-null  object 
 5   Marketing_Airline_Network  35194 non-null  object 
 6   Operating_Airline          35194 non-null  object 
 7   Origin                     35194 non-null  object 
 8   Dest                       35194 non-null  object 
 9   CRSDepTime                 35194 non-null  int32  
 10  DepDel15                   34716 non-null  float64
 11  CRSArrTime                 35194 non-null  int32  
 12  OriginWindDirection        32925 non-null  float64
 13  OriginWindSpeed            35180 non-null  flo

In [228]:
# TESTING PURPOSES
# Get the names of the first 10 columns (0 to 9) 3 and 6 onwards
#columns_to_drop_by_position = label_df.columns[6:]

# Drop these columns by name
#label_df.drop(columns=columns_to_drop_by_position, inplace=True)



In [229]:
label_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35194 entries, 0 to 35193
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       35194 non-null  int32  
 1   Month                      35194 non-null  int32  
 2   DayofMonth                 35194 non-null  int32  
 3   DayOfWeek                  35194 non-null  int32  
 4   FlightDate                 35194 non-null  object 
 5   Marketing_Airline_Network  35194 non-null  object 
 6   Operating_Airline          35194 non-null  object 
 7   Origin                     35194 non-null  object 
 8   Dest                       35194 non-null  object 
 9   CRSDepTime                 35194 non-null  int32  
 10  DepDelay                   34716 non-null  float64
 11  DepDelayMinutes            34716 non-null  float64
 12  DepDel15                   34716 non-null  float64
 13  CRSArrTime                 35194 non-null  int

## Pre-Processing


#### Processing the Nan rows

In [230]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       31316 non-null  int32  
 1   Month                      31316 non-null  int32  
 2   DayofMonth                 31316 non-null  int32  
 3   DayOfWeek                  31316 non-null  int32  
 4   FlightDate                 31316 non-null  object 
 5   Marketing_Airline_Network  31316 non-null  object 
 6   Operating_Airline          31316 non-null  object 
 7   Origin                     31316 non-null  object 
 8   Dest                       31316 non-null  object 
 9   CRSDepTime                 31316 non-null  int32  
 10  DepDel15                   31316 non-null  float64
 11  CRSArrTime                 31316 non-null  int32  
 12  OriginWindDirection        31316 non-null  float64
 13  OriginWindSpeed            31316 non-null  float64


In [231]:
#defining y

label_col = "DepDel15"

y = df[label_col].astype(int)
df["label"] = y
df = df.drop(columns=[label_col])

In [232]:
## Change the CRs dep tiome and arr time to
def hhmm_to_hour(t):
    t = int(t)
    return t // 100

df["DepHour"] = df["CRSDepTime"].apply(hhmm_to_hour)
df["ArrHour"] = df["CRSArrTime"].apply(hhmm_to_hour)

def hhmm_to_minute(t):
    t = int(t)
    return t % 100

df["DepMinute"] = df["CRSDepTime"].apply(hhmm_to_minute)
df["ArrMinute"] = df["CRSArrTime"].apply(hhmm_to_minute)


In [233]:
df = df.drop(columns=["CRSDepTime", "CRSArrTime"])


In [234]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       31316 non-null  int32  
 1   Month                      31316 non-null  int32  
 2   DayofMonth                 31316 non-null  int32  
 3   DayOfWeek                  31316 non-null  int32  
 4   FlightDate                 31316 non-null  object 
 5   Marketing_Airline_Network  31316 non-null  object 
 6   Operating_Airline          31316 non-null  object 
 7   Origin                     31316 non-null  object 
 8   Dest                       31316 non-null  object 
 9   OriginWindDirection        31316 non-null  float64
 10  OriginWindSpeed            31316 non-null  float64
 11  OriginVisibility           31316 non-null  float64
 12  OriginPrecipitation        31316 non-null  object 
 13  OriginClouds               31316 non-null  object 


In [235]:
df["FlightDate"] = pandas.to_datetime(df["FlightDate"])
df["is_weekend"] = df["FlightDate"].dt.dayofweek.isin([5, 6]).astype(int)

In [236]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Year                       31316 non-null  int32         
 1   Month                      31316 non-null  int32         
 2   DayofMonth                 31316 non-null  int32         
 3   DayOfWeek                  31316 non-null  int32         
 4   FlightDate                 31316 non-null  datetime64[ns]
 5   Marketing_Airline_Network  31316 non-null  object        
 6   Operating_Airline          31316 non-null  object        
 7   Origin                     31316 non-null  object        
 8   Dest                       31316 non-null  object        
 9   OriginWindDirection        31316 non-null  float64       
 10  OriginWindSpeed            31316 non-null  float64       
 11  OriginVisibility           31316 non-null  float64       
 12  OriginPre

In [237]:
def arr_to_str(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "None"
    if isinstance(val, (list, np.ndarray)):
        if len(val) == 0:
            return "None"
        return ",".join(map(str, val))
    return str(val)
weather_array_cols = [
    "OriginPrecipitation", "OriginClouds",
    "DestPrecipitation", "DestClouds"
]

for col in weather_array_cols:
    df[col] = df[col].apply(arr_to_str)


In [246]:
df.rename(columns={'Operating_Airline ': 'Operating_Airline'}, inplace=True)

In [247]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Year                       31316 non-null  int32         
 1   Month                      31316 non-null  int32         
 2   DayofMonth                 31316 non-null  int32         
 3   DayOfWeek                  31316 non-null  int32         
 4   FlightDate                 31316 non-null  datetime64[ns]
 5   Marketing_Airline_Network  31316 non-null  object        
 6   Operating_Airline          31316 non-null  object        
 7   Origin                     31316 non-null  object        
 8   Dest                       31316 non-null  object        
 9   OriginWindDirection        31316 non-null  float64       
 10  OriginWindSpeed            31316 non-null  float64       
 11  OriginVisibility           31316 non-null  float64       
 12  OriginPre

In [248]:
y.info()

<class 'pandas.core.series.Series'>
Index: 31316 entries, 0 to 35193
Series name: DepDel15
Non-Null Count  Dtype
--------------  -----
31316 non-null  int64
dtypes: int64(1)
memory usage: 489.3 KB


## SPARK setup


In [249]:
from pyspark.sql import SparkSession
import numpy as np

In [250]:
spark = SparkSession.builder.appName("Pandas to Spark").getOrCreate()
sdf = spark.createDataFrame(df)

In [257]:
sdf.printSchema()
sdf.show()

root
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- DayofMonth: long (nullable = true)
 |-- DayOfWeek: long (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- OriginWindDirection: double (nullable = true)
 |-- OriginWindSpeed: double (nullable = true)
 |-- OriginVisibility: double (nullable = true)
 |-- OriginPrecipitation: string (nullable = true)
 |-- OriginClouds: string (nullable = true)
 |-- DestWindDirection: double (nullable = true)
 |-- DestWindSpeed: double (nullable = true)
 |-- DestVisibility: double (nullable = true)
 |-- DestPrecipitation: string (nullable = true)
 |-- DestClouds: string (nullable = true)
 |-- label: long (nullable = true)
 |-- DepHour: long (nullable = true)
 |-- ArrHour: long (nullable = true)
 |-- DepMinute: long (nullable = true)
 |

In [258]:
cat_cols = [
    "Marketing_Airline_Network",
    "Operating_Airline",
    "Origin",
    "Dest",
    "OriginPrecipitation",
    "OriginClouds",
    "DestPrecipitation",
    "DestClouds"
]

num_cols = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "OriginWindDirection",
    "OriginWindSpeed",
    "OriginVisibility",
    "DestWindDirection",
    "DestWindSpeed",
    "DestVisibility",
    "DepHour",
    "ArrHour",
    "DepMinute",
    "ArrMinute",
    "is_weekend"
]




In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml import Pipeline

# StringIndexers for all categorical features
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# OneHotEncoders for all indexed categorical features
encoders = [
    OneHotEncoder(
        inputCol=c + "_idx",
        outputCol=c + "_ohe"
    )
    for c in cat_cols
]

# VectorAssembler that combines encoded categoricals and numeric features
assembler = VectorAssembler(
    inputCols=[c + "_ohe" for c in cat_cols] + num_cols,
    outputCol="features"
)

# Classifier
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    maxIter=50
)


rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=200,
    maxDepth=15
)


# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

# Train test split
train_df, test_df = sdf.randomSplit([0.8, 0.2], seed=42)

# Fit model
gbt_model = pipeline.fit(train_df)

# Predictions
predictions = gbt_model.transform(test_df)



In [260]:
predictions.select("label", "prediction", "probability").show(50, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.8331459359239942,0.16685406407600578]|
|0    |0.0       |[0.8098347980542003,0.19016520194579967]|
|0    |0.0       |[0.7853909413170314,0.21460905868296865]|
|0    |0.0       |[0.8193954965030851,0.18060450349691493]|
|0    |0.0       |[0.803347553838066,0.196652446161934]   |
|0    |0.0       |[0.8513146004957897,0.14868539950421034]|
|0    |0.0       |[0.8102504967319349,0.18974950326806506]|
|0    |0.0       |[0.9068731466097807,0.09312685339021931]|
|0    |0.0       |[0.8888413768311694,0.11115862316883063]|
|0    |0.0       |[0.8336935368955943,0.16630646310440567]|
|1    |0.0       |[0.8237957280593778,0.17620427194062216]|
|0    |0.0       |[0.7887817141820899,0.21121828581791013]|
|0    |0.0       |[0.8892857384685241,0.1107142615314759] |
|0    |0.0       |[0.8669884592112661,0.

In [261]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_eval.evaluate(predictions)

# F1 (overall)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1:", f1)


Accuracy: 0.7524121013900246
F1: 0.7088730767284422


## Run for best model performance will take a really long time

In [263]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

# Re-define GBTClassifier without featuresCol, as it will be set by the pipeline
gbt_cv = GBTClassifier(labelCol="label")

# Re-create the pipeline including the GBTClassifier
pipeline_cv = Pipeline(stages=indexers + encoders + [assembler, gbt_cv])

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_cv.maxDepth, [3, 5, 7])
    .addGrid(gbt_cv.maxIter, [30, 50, 100])
    .addGrid(gbt_cv.stepSize, [0.05, 0.1, 0.2])
    .addGrid(gbt_cv.maxBins, [32, 64])
    .build()
)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline_cv, # Use the entire pipeline as the estimator
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4     # use multiple cores
)

cv_model = cv.fit(train_df)
best_model = cv_model.bestModel

predictions = best_model.transform(test_df)

KeyboardInterrupt: 